In [1]:
#!/usr/bin/env python3

# Phase 2 — Train Model L (Nepali)
#

In [2]:
# === UPDATE THESE PATHS ===
TRAIN_BIN = "/kaggle/input/notebooks/srinivasu2005/tokenize-nepali/nepali_tokenized/train.bin"
VAL_BIN = "/kaggle/input/notebooks/srinivasu2005/tokenize-nepali/nepali_tokenized/val.bin"
OUTPUT_DIR = "/kaggle/working/checkpoints_nepali"
RESUME_FROM = None  # Set to checkpoint path to resume, e.g. "/kaggle/input/nepali-ckpt/checkpoint_step_5000.pt"

# Model config 
VOCAB_SIZE = 4000
D_MODEL = 512
N_LAYERS = 8
N_HEADS = 8
D_FF = 1184
MAX_SEQ_LEN = 512
DROPOUT = 0.1
TIE_WEIGHTS = True
ROPE_BASE = 10000
PAD_ID = 0; BOS_ID = 2; EOS_ID = 3

# Training config
LR = 3e-4
MIN_LR = 1e-5
WARMUP_STEPS = 2000
MAX_STEPS = 16000
BATCH_SIZE = 16
GRAD_ACCUM = 4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
SEED = 42
EVAL_INTERVAL = 500
SAVE_INTERVAL = 1000
LOG_INTERVAL = 50
EVAL_MAX_BATCHES = 100

In [3]:
import os, sys, math, time, json, random, logging
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s — %(message)s")
logger = logging.getLogger("train_nepali")

In [4]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

[2026-09-03 15:27:06,415] INFO — Device: cuda


In [5]:
@dataclass
class TransformerConfig:
    vocab_size: int = VOCAB_SIZE
    d_model: int = D_MODEL
    n_layers: int = N_LAYERS
    n_heads: int = N_HEADS
    d_ff: int = D_FF
    max_seq_len: int = MAX_SEQ_LEN
    dropout: float = DROPOUT
    tie_weights: bool = TIE_WEIGHTS
    rope_base: int = ROPE_BASE
    pad_id: int = PAD_ID; bos_id: int = BOS_ID; eos_id: int = EOS_ID
    
    @property
    def d_k(self):
        return self.d_model // self.n_heads

config = TransformerConfig()

In [6]:
def precompute_rope_frequencies(d_k, max_seq_len, base=10000, device=None):
    freq_indices = torch.arange(0, d_k, 2, dtype=torch.float32, device=device)
    theta = 1.0 / (base ** (freq_indices / d_k))
    positions = torch.arange(0, max_seq_len, dtype=torch.float32, device=device)
    angles = torch.outer(positions, theta)
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos
    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

In [7]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads, self.d_k, self.d_model = cfg.n_heads, cfg.d_k, cfg.d_model
        self.W_Q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_K = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_V = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_O = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        cos, sin = precompute_rope_frequencies(cfg.d_k, cfg.max_seq_len, cfg.rope_base)
        self.register_buffer("cos_table", cos)
        self.register_buffer("sin_table", sin)
        mask = torch.triu(torch.ones(cfg.max_seq_len, cfg.max_seq_len), diagonal=1).bool()
        self.register_buffer("causal_mask", mask)

    def forward(self, x, return_attention=False):
        B, T, d = x.shape
        Q = self.W_Q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        cos, sin = self.cos_table[:T], self.sin_table[:T]
        Q, K = apply_rope(Q, cos, sin), apply_rope(K, cos, sin)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(self.causal_mask[:T, :T].unsqueeze(0).unsqueeze(0), float('-inf'))
        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights_d = self.attn_dropout(attn_weights)
        context = torch.matmul(attn_weights_d, V)
        context = context.transpose(1, 2).contiguous().view(B, T, self.d_model)
        output = self.resid_dropout(self.W_O(context))
        return (output, attn_weights) if return_attention else (output, None)

In [8]:
class SwiGLUFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.W_gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.dropout(self.W_down(F.silu(self.W_gate(x)) * self.W_up(x)))

In [9]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = MultiHeadCausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = SwiGLUFeedForward(cfg)
    def forward(self, x, return_attention=False):
        attn_out, attn_w = self.attn(self.ln1(x), return_attention)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, attn_w

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.emb_dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_ln = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_weights:
            self.lm_head.weight = self.embedding.weight
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None: torch.nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.padding_idx is not None: m.weight.data[m.padding_idx].zero_()
            elif isinstance(m, nn.LayerNorm):
                torch.nn.init.ones_(m.weight); torch.nn.init.zeros_(m.bias)
    
    def forward(self, input_ids, targets=None, return_attention=False):
        x = self.emb_dropout(self.embedding(input_ids))
        all_attn = [] if return_attention else None
        for block in self.blocks:
            x, aw = block(x, return_attention)
            if return_attention and aw is not None: all_attn.append(aw)
        logits = self.lm_head(self.final_ln(x))
        result = {"logits": logits}
        if targets is not None:
            result["loss"] = F.cross_entropy(logits.view(-1, self.config.vocab_size), targets.view(-1), ignore_index=self.config.pad_id)
        if return_attention: result["attention_weights"] = all_attn
        return result

In [10]:
class TokenizedDataset(Dataset):
    def __init__(self, bin_path, seq_len=512):
        self.seq_len = seq_len
        self.data = np.memmap(bin_path, dtype=np.uint16, mode='r')
        self.n_samples = max(0, (len(self.data) - 1) // seq_len)
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        start = idx * self.seq_len
        chunk = self.data[start: start + self.seq_len + 1].astype(np.int64)
        return torch.from_numpy(chunk[:-1]), torch.from_numpy(chunk[1:])

In [11]:
model = DecoderOnlyTransformer(config).to(device)
total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
logger.info(f"Model: {train_p/1e6:.2f}M trainable, {total_p/1e6:.2f}M total")

[2026-09-03 15:27:07,298] INFO — Model: 25.00M trainable, 25.00M total


In [12]:
logger.info("Running causal mask verification...")
model.eval()
x1 = torch.tensor([[10, 20, 30, 40, 50]], device=device)
x2 = torch.tensor([[10, 20, 30, 40, 70]], device=device)
with torch.no_grad():
    l1 = model(x1)["logits"][0, 3, :]
    l2 = model(x2)["logits"][0, 3, :]
diff = (l1 - l2).abs().max().item()
assert diff < 1e-5, f"Causal mask FAILED! diff={diff}"
logger.info(f"  ✅ Causal mask verified (max diff={diff:.2e})")
model.train()

[2026-09-03 15:27:07,312] INFO — Running causal mask verification...
[2026-09-03 15:27:07,740] INFO —   ✅ Causal mask verified (max diff=0.00e+00)


DecoderOnlyTransformer(
  (embedding): Embedding(4000, 512, padding_idx=0)
  (emb_dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-7): 8 x TransformerBlock(
      (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadCausalSelfAttention(
        (W_Q): Linear(in_features=512, out_features=512, bias=False)
        (W_K): Linear(in_features=512, out_features=512, bias=False)
        (W_V): Linear(in_features=512, out_features=512, bias=False)
        (W_O): Linear(in_features=512, out_features=512, bias=False)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ffn): SwiGLUFeedForward(
        (W_gate): Linear(in_features=512, out_features=1184, bias=False)
        (W_up): Linear(in_features=512, out_features=1184, bias=False)
        (W_down): Linear(in_features=1184, out_features=512, bias=False)
 

In [13]:
train_ds = TokenizedDataset(TRAIN_BIN, MAX_SEQ_LEN)
val_ds = TokenizedDataset(VAL_BIN, MAX_SEQ_LEN)
logger.info(f"Train: {train_ds.n_samples:,} samples ({len(train_ds.data):,} tokens)")
logger.info(f"Val:   {val_ds.n_samples:,} samples ({len(val_ds.data):,} tokens)")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, drop_last=True)

[2026-09-03 15:27:07,794] INFO — Train: 1,695,895 samples (868,298,607 tokens)
[2026-09-03 15:27:07,794] INFO — Val:   35,289 samples (18,068,301 tokens)


In [14]:
decay_params, no_decay_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad: continue
    if p.ndim < 2 or "ln" in name or "bias" in name: no_decay_params.append(p)
    else: decay_params.append(p)
optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0}
], lr=LR, betas=(0.9, 0.95), eps=1e-8)

def lr_lambda(step):
    if step < WARMUP_STEPS: return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    decay = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR / LR + (1.0 - MIN_LR / LR) * decay
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda')

In [15]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
train_logs = []

def log_entry(step, **kwargs):
    entry = {"step": step, "time": time.time(), **kwargs}
    train_logs.append(entry)
    with open(os.path.join(OUTPUT_DIR, "training_log.json"), "w") as f:
        json.dump(train_logs, f)

In [16]:
@torch.no_grad()
def evaluate(max_batches=EVAL_MAX_BATCHES):
    model.eval()
    total_loss, n = 0.0, 0
    for i, (x, y) in enumerate(val_loader):
        if i >= max_batches: break
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            result = model(x, targets=y)
        total_loss += result["loss"].item(); n += 1
    model.train()
    avg = total_loss / max(1, n)
    return avg, math.exp(avg) if avg < 20 else float('inf')

In [17]:
start_step = 0
best_val_loss = float('inf')
if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "scaler_state_dict" in ckpt: scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_step = ckpt.get("step", 0)
    best_val_loss = ckpt.get("val_loss", float('inf'))
    logger.info(f"Resumed from step {start_step}, best_val_loss={best_val_loss:.4f}")

In [18]:
model.train()
step = start_step
accum_loss, accum_n = 0.0, 0
train_iter = iter(train_loader)
start_time = time.time()

logger.info(f"Training: steps {start_step}→{MAX_STEPS}, eff_batch={BATCH_SIZE*GRAD_ACCUM}")

while step < MAX_STEPS:
    optimizer.zero_grad()
    for _ in range(GRAD_ACCUM):
        try: x, y = next(train_iter)
        except StopIteration: train_iter = iter(train_loader); x, y = next(train_iter)
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            result = model(x, targets=y)
            loss = result["loss"] / GRAD_ACCUM
        scaler.scale(loss).backward()
        accum_loss += result["loss"].item(); accum_n += 1
    
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer); scaler.update(); scheduler.step()
    step += 1
    
    if step % LOG_INTERVAL == 0:
        avg = accum_loss / max(1, accum_n)
        elapsed = time.time() - start_time
        tps = (step - start_step) * BATCH_SIZE * GRAD_ACCUM * MAX_SEQ_LEN / elapsed
        cur_lr = scheduler.get_last_lr()[0]
        ppl_str = f"{math.exp(avg):.1f}" if avg < 20 else "inf"
        logger.info(f"Step {step}/{MAX_STEPS} | Loss {avg:.4f} | PPL {ppl_str} | LR {cur_lr:.2e} | {tps:.0f} tok/s")
        log_entry(step, train_loss=avg, lr=cur_lr, tok_per_sec=tps)
        accum_loss, accum_n = 0.0, 0
    
    is_best = False
    if step % EVAL_INTERVAL == 0:
        vl, vp = evaluate()
        logger.info(f"  Val: loss={vl:.4f}, ppl={vp:.1f}")
        log_entry(step, val_loss=vl, val_ppl=vp)
        if vl < best_val_loss:
            best_val_loss = vl
            is_best = True
    
    if step % SAVE_INTERVAL == 0 or is_best:
        ckpt = {"step": step, "val_loss": best_val_loss,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "config": config.__dict__}
        
        if step % SAVE_INTERVAL == 0:
            torch.save(ckpt, os.path.join(OUTPUT_DIR, f"checkpoint_step_{step}.pt"))
            logger.info(f"  Saved checkpoint_step_{step}.pt")
            
        if is_best:
            torch.save(ckpt, os.path.join(OUTPUT_DIR, "best_model.pt"))
            logger.info(f"  Saved new best_model.pt (val_loss={best_val_loss:.4f})")

[2026-09-03 15:27:12,749] INFO — Training: steps 0→16000, eff_batch=64
[2026-09-03 15:28:26,192] INFO — Step 50/16000 | Loss 8.3165 | PPL 4090.7 | LR 7.50e-06 | 22308 tok/s
[2026-09-03 15:29:31,987] INFO — Step 100/16000 | Loss 7.8354 | PPL 2528.5 | LR 1.50e-05 | 23534 tok/s
[2026-09-03 15:30:33,106] INFO — Step 150/16000 | Loss 7.4715 | PPL 1757.2 | LR 2.25e-05 | 24532 tok/s
[2026-09-03 15:31:32,044] INFO — Step 200/16000 | Loss 7.1220 | PPL 1238.9 | LR 3.00e-05 | 25275 tok/s
[2026-09-03 15:32:29,691] INFO — Step 250/16000 | Loss 6.8839 | PPL 976.4 | LR 3.75e-05 | 25847 tok/s
[2026-09-03 15:33:26,190] INFO — Step 300/16000 | Loss 6.7419 | PPL 847.1 | LR 4.50e-05 | 26324 tok/s
[2026-09-03 15:34:21,761] INFO — Step 350/16000 | Loss 6.5792 | PPL 720.0 | LR 5.25e-05 | 26733 tok/s
[2026-09-03 15:35:16,670] INFO — Step 400/16000 | Loss 6.3896 | PPL 595.6 | LR 6.00e-05 | 27085 tok/s
[2026-09-03 15:36:11,286] INFO — Step 450/16000 | Loss 6.1933 | PPL 489.5 | LR 6.75e-05 | 27381 tok/s
[2026-09

In [19]:
vl, vp = evaluate()
is_best = vl < best_val_loss
ckpt = {"step": step, "val_loss": vl,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "config": config.__dict__}
torch.save(ckpt, os.path.join(OUTPUT_DIR, f"checkpoint_step_{step}.pt"))
if is_best: torch.save(ckpt, os.path.join(OUTPUT_DIR, "best_model.pt"))

In [20]:
elapsed = time.time() - start_time
logger.info(f"\n{'='*60}")
logger.info(f"Nepali Training Complete!")
logger.info(f"  Steps: {MAX_STEPS}, Time: {elapsed/3600:.1f}h")
logger.info(f"  Final val loss: {vl:.4f}, PPL: {vp:.1f}")
logger.info(f"  Best val loss: {best_val_loss:.4f}")
logger.info(f"  Checkpoints: {OUTPUT_DIR}/")
logger.info(f"{'='*60}")

[2026-09-03 20:14:45,558] INFO — 
[2026-09-03 20:14:45,560] INFO — Nepali Training Complete!
[2026-09-03 20:14:45,560] INFO —   Steps: 16000, Time: 4.8h
[2026-09-03 20:14:45,561] INFO —   Final val loss: 3.1606, PPL: 23.6
[2026-09-03 20:14:45,562] INFO —   Best val loss: 3.1606
[2026-09-03 20:14:45,562] INFO —   Checkpoints: /kaggle/working/checkpoints_nepali/
[2026-09-03 20:14:45,563] INFO — ============================================================
